# English STELLA Transcriptions Dataset

The STELLA dataset...


# Data preparation

Format Dataset into the wanted architecture. This procedure extracts audiobook transcriptions from the original dataset and sorts them into the same splits as the audio files.

```
txt
├── LANG
│   ├── HOUR_SPLIT
│   │   ├── SECTION_SPLIT
│   │   │   ├── books.txt
│   │   │   ├── meta.json
│   │   │   └── transcription.txt
│   │   ├── ...
│   ├── ...
│   ...

```
- txt : folder containing transcriptions
- LANG: corresponds to the given language
- HOUR_SPLIR: corresponds to the size of the section splits in number of hours of speech,
              formatted as (50h, 100h, ..., 3200h)
- SECTION_SPLIT: separation of content into sections with equal amount of speech content.
- books.txt: the list of books used for this split
- meta.json: metadata generated during clean-up used to measure effectiveness of cleaning.
- transcript.txt: the agregated transcripts of the audiobooks in the list.

In [1]:
import platform

from lexical_benchmark import settings
from lexical_benchmark.datasets import stella
from lexical_benchmark.utils import timed_status

assert (
    platform.node() in settings.PATH.KNOWN_HOSTS
), "Not in a known device, must provide custom PATH locations"

prep = stella.STELAPrepTranscripts(lang="EN")
with timed_status(status="Preping stela transcriptions...", complete_status="Succesfuly build STELA Transcript dataset !"):
    prep.build_transcript()

Output()

Succesfuly build STELA Transcript dataset ! (Total time: 7 minutes and 22 seconds)

## Data Cleaning

Clean-up text to keep only clean words that can be piped through the dictionairy.

RULES (Order Matters):
1) Illustration tag removal
2) URL removal
3) TextNormalisation : correct accents & remove non-printable characters
4) Trancribe numbers
5) Remove roman numerals
6) Fix symbols ($,€, etc..)
7) AZFilter

    * replace '-' with a space to extract hyphenated words (fifty-five -> fifty five)

    * Keeps apostrophe char(*'*) to protect shorthands (ex: ain't)
  
    * purges everything not between [A-Z].

    * lowecases everything
8) Fix words by removing prefix and trailing quote char (')

In [2]:
import platform

from lexical_benchmark import settings
from lexical_benchmark.datasets import stella, utils as dataset_utils
from lexical_benchmark.utils import timed_status

assert (
    platform.node() in settings.PATH.KNOWN_HOSTS
), "Not in a known device, must provide custom PATH locations"

dataset = stella.STELATranscriptDataset()
with timed_status(status="Pre-processing STELA/EN Transcripts", complete_status="Succesfuly pre-processed up STELA/EN Transcripts!"):
    dataset_utils.DatasetCleaner.cleanup_files(
        filemap=dataset.raw2clean_filesmap("EN"),
        ruleset=dataset.clean_up_rules("EN"),
        save_logs=True
    )

Output()

Succesfuly pre-processed up STELA/EN Transcripts! (Total time: 54 minutes and 25 seconds)

# Word Filtering

Using a pre-selected lexicon we filter the corpus to separated known from unknown words

In [2]:
import platform

from lexical_benchmark import settings
from lexical_benchmark.datasets import stella
from lexical_benchmark.datasets import utils as dataset_utils
from lexical_benchmark.utils import timed_status

assert (
    platform.node() in settings.PATH.KNOWN_HOSTS
), "Not in a known device, must provide custom PATH locations"

dataset = stella.STELATranscriptDataset()

with timed_status(status="Word Filtering", complete_status="Succesfuly completed word filtering !"):
    dataset_utils.DatasetCleaner.word_validate_files(
        filemap=dataset.word_validation_filesmap("EN"),
        cleaner=dataset_utils.DictionairyCleaner(lang="EN"),
    )

Output()

Succesfuly completed word filtering ! (Total time: 2 minutes and 16 seconds)

## Compute word frequency maps

To allow statistics on word cleaning we generate word_frequency table for all steps of the cleaning process :

1) raw transcription frequency maps
2) unvalidated clean transcriptions frequency maps
3) clean transcription frequency maps


In [1]:
import platform

from lexical_benchmark import settings
from lexical_benchmark.datasets import stella
from lexical_benchmark.utils import timed_status

assert (
    platform.node() in settings.PATH.KNOWN_HOSTS
), "Not in a known device, must provide custom PATH locations"

dataset = stella.STELATranscriptDataset()

with timed_status(status="Computing word frequencies of units...", complete_status="Succesfully computed all word frequencies !"):
    dataset.build_clean_word_frequencies()
    dataset.build_rejected_word_frequencies()
    dataset.build_preprocess_word_frequencies()

Output()

Succesfully computed all word frequencies ! (Total time: 7 minutes and 3 seconds)

Global Average STATS

Average on each block inside each hour split. EN/50h is the result of the average of al EN/50h/XX blocks.

,section,Token RJ (Avg per block),Tokens (Avg per block),Types (Avg per block),Type RJ (Avg per block)
0,EN/50h,0.59%,"570,279","20,403",3.36%
1,EN/100h,0.64%,"1,060,532","29,267",6.04%
2,EN/200h,0.67%,"2,358,527","42,324",11.52%
3,EN/400h,0.78%,"4,655,191","58,253",22.97%
4,EN/800h,1.00%,"8,810,671","75,258",35.16%
5,EN/1600h,1.71%,"20,108,147","102,994",48.26%
6,EN/3200h,1.51%,"41,395,136","132,201",60.66%
